In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

groups = pd.read_csv("data/groups.csv")
departures = pd.read_csv("data/departures.csv")
candidate = ['orders_in_trip', 'max_angle_deg', 'spread_km', 'far_km', 'path_km',
             'path_over_far', 'far_first_ratio', 'road_savings',
             'ready_orders', 'available_couriers', 'load_ratio', 'hour']
target_col = "label_A"    # опоздала ли группа
day_col = "day_id"        # для честного сплита по дням

In [2]:
X = groups[candidate].values
y = groups['label_a'].values
days = groups['day_id'].values

gss = GroupShuffleSplit(n_splits=2, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups=days))
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

mu = X_train.mean(axis=0)
sigma = X_train.std(axis=0)
X_norm_train = (X_train - mu) / sigma
X_norm_val = (X_val - mu) / sigma

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(X_norm_train.shape[1])
b = 0.0
lr = 0.1
epochs = 1000
m = len(y_train)

for epoch in range(epochs):
    z = X_norm_train @ w + b
    y_hat = sigmoid(z)
    error = y_hat - y_train
    grad_w = (1/m) * (X_norm_train.T @ error)
    grad_b = (1/m) * np.sum(error)
    w -= lr * grad_w
    b -= lr * grad_b

y_hat_val_plain = sigmoid(X_norm_val @ w + b)
print(roc_auc_score(y_val, y_hat_val_plain))
K = int(y_val.sum())
print(K)
order = np.argsort(y_hat_val_plain)
y_val[order[-K:]].mean()

In [ ]:
# поправка на дисбаланс: вес класса 1 обратно пропорционален его доле
pos_rate = y_train.mean()
class_weight = np.where(y_train == 1, 1/pos_rate, 1/(1-pos_rate))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

w = np.zeros(X_norm_train.shape[1])
b = 0.0
lr = 0.1
epochs = 1000
m = len(y_train)

for epoch in range(epochs):
    z = X_norm_train @ w + b
    y_hat = sigmoid(z)
    error = (y_hat - y_train) * class_weight     # веса добавлены прямо в error
    grad_w = (1/m) * (X_norm_train.T @ error)
    grad_b = (1/m) * np.sum(error)
    w -= lr * grad_w
    b -= lr * grad_b

y_hat_val_weighted = sigmoid(X_norm_val @ w + b)
print(roc_auc_score(y_val, y_hat_val_weighted))
K = int(y_hat_val_weighted.sum())
print(K)
order = np.argsort(y_hat_val_weighted)
y_val[order[-K:]].mean()


In [3]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [4]:
def train(X_norm, y, lr=0.1, epochs=1000, use_weights=False):
    m = len(y)
    w = np.zeros(X_norm.shape[1])
    b = 0.0

    if use_weights:
        pos_rate = y_tr.mean()
        cw = np.where(y == 1, 1/pos_rate, 1/(1 - pos_rate))
    else:
        cw = np.ones(m)          # все веса равны 1: ничего не меняется

    for epoch in range(epochs):
        z = X_norm @ w + b
        y_hat = sigmoid(z)
        error = (y_hat - y) * cw
        grad_w = (1/m) * (X_norm.T @ error)
        grad_b = (1/m) * np.sum(error)
        w -= lr * grad_w
        b -= lr * grad_b
        return w, b

In [17]:
gkf = GroupKFold(n_splits=5)

total_need = 0
total_rules_flags = 0
results = []
for train_idx, val_idx in gkf.split(X, y, groups=days):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]

    mu = X_tr.mean(axis=0)
    sigma = X_tr.std(axis=0)

    X_tr_norm = (X_tr - mu) / sigma
    X_va_norm = (X_va - mu) / sigma


    w, b = train(X_tr_norm, y_tr)
    y_hat_va = sigmoid(X_va_norm @ w + b)

    det_va = groups['det_any_geo'].values[val_idx]
    N = int(det_va.sum())
    hits_rules = ((det_va == 1) & (y_va == 1)).sum()
    order = np.argsort(y_hat_va)


    hits_model = y_va[order[-N:]].sum()
    results.append((hits_rules, hits_model))



    hr = ((det_va == 1) & (y_va == 1)).sum()
    order = np.argsort(y_hat_va)[::-1]
    cum = np.cumsum(y_va[order])
    need = np.argmax(cum >= hr) + 1
    total_need += need
    total_rules_flags += det_va.sum()

print(results)
wins = sum(m > r for r, m in results)
print("модель выиграла на", wins, "из", len(results))
print(total_need, total_rules_flags)

[(np.int64(23), np.int64(30)), (np.int64(19), np.int64(40)), (np.int64(22), np.int64(42)), (np.int64(20), np.int64(17)), (np.int64(10), np.int64(23))]
модель выиграла на 4 из 5
234 386


In [5]:
#проверка на переобученность
gkf = GroupKFold(n_splits=5)

total_need = 0
total_rules_flags = 0
results = []
pizzerias = groups['pizzeria'].values
for train_idx, val_idx in gkf.split(X, y, groups=pizzerias):
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[val_idx], y[val_idx]

    mu = X_tr.mean(axis=0)
    sigma = X_tr.std(axis=0)

    X_tr_norm = (X_tr - mu) / sigma
    X_va_norm = (X_va - mu) / sigma

    w, b = train(X_tr_norm, y_tr)
    y_hat_va = sigmoid(X_va_norm @ w + b)

    det_va = groups['det_any_geo'].values[val_idx]
    N = int(det_va.sum())
    hits_rules = ((det_va == 1) & (y_va == 1)).sum()
    order = np.argsort(y_hat_va)

    hits_model = y_va[order[-N:]].sum()
    results.append((hits_rules, hits_model))

    hr = ((det_va == 1) & (y_va == 1)).sum()
    order = np.argsort(y_hat_va)[::-1]
    cum = np.cumsum(y_va[order])
    need = np.argmax(cum >= hr) + 1
    total_need += need
    total_rules_flags += det_va.sum()

print(results)
wins = sum(m > r for r, m in results)
print("модель выиграла на", wins, "из", len(results))
print(total_need, total_rules_flags)

[(np.int64(9), np.int64(21)), (np.int64(29), np.int64(40)), (np.int64(19), np.int64(42)), (np.int64(21), np.int64(32)), (np.int64(16), np.int64(18))]
модель выиграла на 5 из 5
241 386


In [ ]:
print((y_hat_val_plain > 0.5).sum(), (y_hat_val_weighted > 0.5).sum())

In [ ]:
hits_plain = ((y_hat_val_plain > 0.5) & (y_val == 1)).sum()
hits_weighted = ((y_hat_val_weighted > 0.5) & (y_val == 1)).sum()
print(hits_plain, hits_weighted)

In [ ]:
det_val = groups['det_any_geo'].values[val_idx]
print(det_val.sum())                      # сколько групп правила назвали плохими
print(((det_val == 1) & (y_val == 1)).sum())   # сколько из них правда плохие

order = np.argsort(y_hat_val_plain)
y_val[order[-83:]].sum()